In [ ]:
from example_mortality_data import MALE_QX, FEMALE_QX

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import simpson
from scipy.special import comb
import itertools
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
"""
This script determines the fair participation rate (pi) for an 'outlier' 
joining a tontine cohort of a different age. 

Main factor examined is the outlier's age.

It balances the pool by ensuring that the Expected Present Value (EPV) of 
payouts is actuarially neutral for both the outlier and the cohort. It 
calculates survival probabilities, models stochastic pool exhaustion using 
binomial distributions, and uses an iterative solver to find the equilibrium 
participation rate. Results are mapped onto a 3D surface for sensitivity analysis.

Replicates results from Milevsky & Salisbury (2016), "Equitable Retirement Income Tontines: Mixing Cohorts Without Discriminating." ASTIN Bulletin 46(3), 571-604. https://ssrn.com/abstract/2700362
"""

# --- Mortality Table Preprocessing ---

# Derive survival curves (lx) from mortality rate tables (qx)
# lx represents the number of survivors at each age relative to the starting population
max_age = len(MALE_QX)
lx_male = np.ones(max_age + 1)
lx_female = np.ones(max_age + 1)

for i in range(max_age):
    lx_male[i+1] = lx_male[i] * (1.0 - MALE_QX[i])
    lx_female[i+1] = lx_female[i] * (1.0 - FEMALE_QX[i])

# --- Probability Functions ---
def tpx(age, t, gender='male'):
    """
    Computes the probability of an individual aged 'age' surviving 't' more years.
    Uses linear interpolation (np.interp) to handle non-integer ages or time steps.
    """
    lx = lx_male if gender == 'male' else lx_female
    
    # Baseline survival constant for the starting age
    l_age = np.interp(age, np.arange(max_age + 1), lx)
    
    # Boundary check for ages beyond the table limits
    if l_age <= 0:
        return np.zeros_like(t) if isinstance(t, np.ndarray) else 0.0
        
    target_ages = age + t
    l_target = np.interp(target_ages, np.arange(max_age + 1), lx)
    
    return l_target / l_age

# --- Actuarial Fairness Solver ---
def calculate_outlier_pi_by_age(n_cohort, age_cohort, age_outlier, interest_rate=0.03):
    """
    Finds the fair participation rate (pi) for an 'outlier' joining a homogeneous cohort.
    The goal is to ensure the expected present value (EPV) of benefits is proportional
    to the expected value of their contributions.
    """
    steps = 100  # Numerical integration resolution
    times = np.linspace(0, 100, steps)
    
    n_arr = np.array([n_cohort, 1]) # Cohort size vs 1 outlier
    invested_arr = np.array([1.0, 1.0])
    ages = [age_cohort, age_outlier]
    invested_total = np.sum(invested_arr * n_arr)

    # Calculate survival vectors and life expectancy (annuity) factors for both parties
    tpx_matrix = np.array([tpx(age, times) for age in ages])
    discount = np.exp(-interest_rate * times)
    ax_arr = np.array([simpson(y=discount * tpx_matrix[i], x=times) for i in range(2)])
    
    # Define mortality density d(t): the rate at which the total pool releases funds
    d_t = np.zeros_like(times)
    for j in range(2):
        alpha_j = (invested_arr[j] * n_arr[j]) / invested_total
        d_t += alpha_j * (1.0 / ax_arr[j]) * tpx_matrix[j]

    def calculate_epv(pi_vals):
        """
        Calculates the expected present value (epv) for each party under current participation rates.
        Uses binomial distributions to account for the stochastic survival of the pool.
        """
        epv = np.zeros(2)
        prob_matrix = []
        for j in range(2):
            p = tpx_matrix[j]
            n_j = n_arr[j]
            # Probabilities of exactly k survivors in each group at time t
            cohort_probs = [comb(n_j, k) * (p**k) * ((1-p)**(n_j-k)) for k in range(n_j + 1)]
            prob_matrix.append(cohort_probs)

        ranges = [range(n_j + 1) for n_j in n_arr]

        for i in range(2):
            expected_share_t = np.zeros_like(times)
            # Iterate through all possible survivor states (Cartesian product)
            for state in itertools.product(*ranges):
                if state[i] == 0: continue # No payout if the individual is dead
                
                prob_state_t = np.ones_like(times)
                pool_value_t = 0.0

                for j in range(2):
                    k = state[j]
                    if j == i:
                        # Conditional probability: individual i is one of the k survivors
                        prob_state_t *= prob_matrix[j][k] * (k / n_arr[j])
                    else:
                        prob_state_t *= prob_matrix[j][k]
                    pool_value_t += pi_vals[j] * invested_arr[j] * k

                pool_value_t = np.where(pool_value_t == 0, 1e-9, pool_value_t)
                expected_share_t += prob_state_t * (pi_vals[i] / pool_value_t)

            integrand = discount * invested_total * d_t * expected_share_t
            epv[i] = simpson(y=integrand, x=times)
        return epv

    # Iterative optimization to find the pi where the outlier's payout is actuarially fair
    pi = np.array([1.0, 1.0]) 
    etas = [0.1, 0.01, 0.001, 0.0001] # Step sizes for the gradient-style search
    tolerance = 1e-8 
    max_iter_per_eta = 10_000 

    for eta in etas:
        for _ in range(max_iter_per_eta):
            epv_current = calculate_epv(pi)
            # The target is an equitable distribution of the total pool value
            target = np.sum((invested_arr * n_arr / invested_total) * epv_current)
            
            if np.abs(epv_current[1] - target) < tolerance:
                break
                
            # If the outlier's expected return is too low, increase their participation rate
            if epv_current[1] < target:
                pi[1] += eta
            else:
                break 
                
    return pi[1] / pi[0]

# --- Sensitivity Analysis (Grid Sweep) ---
# Sweep over different cohort sizes and outlier's ages to see how pi evolves
fixed_cohort_age = 60
cohort_sizes = np.arange(5, 31, 1)  
outlier_ages = np.arange(60, 91, 1) 

X, Y = np.meshgrid(cohort_sizes, outlier_ages)
Z = np.zeros_like(X, dtype=float)

print(f"Sweeping {X.size} grid points...")
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = calculate_outlier_pi_by_age(X[i, j], fixed_cohort_age, Y[i, j])

# --- Visualization ---
plt.style.use('dark_background')

fig = plt.figure(figsize=(14, 9))
ax = fig.add_subplot(111, projection='3d')

# Render the result as a 3D surface to visualize the "Fairness Landscape"
surf = ax.plot_surface(X, Y, Z, cmap='plasma', edgecolor='navy', lw=0.8)

ax.set_xlabel('Cohort Size (members)', labelpad=10)
ax.set_ylabel("Outlier's Age (years)", labelpad=10)
ax.set_zlabel("Outlier's participation rate (pi)", labelpad=10)
ax.set_title(f'Participation (pi) Rates for Single Outlier vs Age {fixed_cohort_age} Cohort', fontsize=14)

fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10)
plt.show()

In [ ]:
"""
This script determines the fair participation rate (pi) for an 'outlier' 
joining a tontine cohort with a different investment amount. 

Main factor examined is the outlier's investment sum.

It balances the pool by ensuring that the Expected Present Value (EPV) of 
payouts is actuarially neutral for both the outlier and the cohort. It 
calculates survival probabilities, models stochastic pool exhaustion using 
binomial distributions, and uses an iterative solver to find the equilibrium 
participation rate. Results are mapped onto a 3D surface for sensitivity analysis.

Replicates results from Milevsky & Salisbury (2016), "Equitable Retirement Income Tontines: Mixing Cohorts Without Discriminating." ASTIN Bulletin 46(3), 571-604. https://ssrn.com/abstract/2700362
"""

# --- Data Initialization & Actuarial Functions ---

# Convert annual mortality rates (QX) into a cumulative survival curve S(x)
# This serves as the baseline for all survival probability lookups
px_arr = 1.0 - np.array(MALE_QX)
S_curve = np.cumprod(np.insert(px_arr, 0, 1.0)) 
ages_array = np.arange(len(S_curve))

def tpx(age, t):
    """
    Computes the probability that an individual of a given 'age' survives for 't' years.
    Uses linear interpolation on the empirical S_curve to handle fractional years.
    """
    target_ages = age + t
    S_t = np.interp(target_ages, ages_array, S_curve, right=0.0)
    return S_t / S_curve[int(age)]

# --- Tontine Fairness Solver ---

def calculate_outlier_pi_by_investment(n_cohort, w_outlier, age_c, age_o, interest_rate=0.03):
    """
    Calculates the relative participation rate (pi) required to maintain 
    actuarial fairness between a homogeneous cohort and a single outlier.
    """
    steps = 100  
    times = np.linspace(0, 100, steps)
    
    # Define demographic and financial vectors for the two groups
    n_arr = np.array([n_cohort, 1])
    invested_arr = np.array([1.0, float(w_outlier)])
    ages = [age_c, age_o]
    invested_total = np.sum(invested_arr * n_arr)

    # Pre-compute time-dependent survival probabilities and net single premium annuities
    tpx_matrix = np.array([tpx(age, times) for age in ages])
    discount = np.exp(-interest_rate * times)
    ax_arr = np.array([simpson(y=discount * tpx_matrix[i], x=times) for i in range(2)])
    
    # Compute the aggregate mortality density d(t) of the entire pool
    d_t = np.zeros_like(times)
    for j in range(2):
        alpha_j = (invested_arr[j] * n_arr[j]) / invested_total
        d_t += alpha_j * (1.0 / ax_arr[j]) * tpx_matrix[j]

    def calculate_epv(pi_vals):
        """
        Computes the expected present value of payouts for each group 
        given a specific set of participation rates (pi).
        """
        epv = np.zeros(2)
        prob_matrix = []
        # Calculate binomial distribution of survivors for both groups at each time step
        for j in range(2):
            p, n_j = tpx_matrix[j], n_arr[j]
            cohort_probs = [comb(n_j, k) * (p**k) * ((1-p)**(n_j-k)) for k in range(n_j + 1)]
            prob_matrix.append(cohort_probs)

        ranges = [range(n_j + 1) for n_j in n_arr]

        # Use Cartesian product to iterate through all possible survival states of the pool
        for i in range(2):
            expected_share_t = np.zeros_like(times)
            for state in itertools.product(*ranges):
                if state[i] == 0: continue
                
                prob_state_t = np.ones_like(times)
                pool_value_t = 0.0
                for j in range(2):
                    k = state[j]
                    if j == i:
                        # Individual probability of survival weighted by group share
                        prob_state_t *= prob_matrix[j][k] * (k / n_arr[j])
                    else:
                        prob_state_t *= prob_matrix[j][k]
                    pool_value_t += pi_vals[j] * invested_arr[j] * k

                pool_value_t = np.where(pool_value_t == 0, 1e-9, pool_value_t)
                expected_share_t += prob_state_t * (pi_vals[i] / pool_value_t)

            # Integrate discounted expected cash flows
            integrand = discount * invested_total * d_t * expected_share_t
            epv[i] = simpson(y=integrand, x=times)
        return epv

    # Iterative optimization: Adjust pi values until the expected value for 
    # both groups converges to the fair market target.
    pi = np.array([1.0, 1.0]) 
    etas = [0.1, 0.01, 0.001, 0.0001] # Step sizes for gradient descent refinement
    tolerance = 1e-7 
    max_iter_per_eta = 1000 

    for eta in etas:
        for _ in range(max_iter_per_eta):
            epv_current = calculate_epv(pi)
            if np.abs(epv_current[0] - epv_current[1]) < tolerance:
                break
                
            target = np.sum((invested_arr * n_arr / invested_total) * epv_current)
            improved = False
            for i in range(2):
                if epv_current[i] < target:
                    pi_test = pi.copy()
                    pi_test[i] += eta
                    epv_test = calculate_epv(pi_test)
                    target_test = np.sum((invested_arr * n_arr / invested_total) * epv_test)
                    if epv_test[i] < target_test:
                        pi = pi_test
                        improved = True
            if not improved: break
                
    return pi[1] / pi[0]

# --- Parameter Sweep & Visualization ---

age_cohort = 65
age_outlier = 75

# Define the grid: vary cohort population and the outlier's capital contribution
cohort_sizes = np.arange(10, 31, 1)        
outlier_investments = np.arange(1, 11, 1) 

X, Y = np.meshgrid(cohort_sizes, outlier_investments)
Z = np.zeros_like(X, dtype=float)

# Execute the sweep across the 2D parameter space
print(f"Sweeping {X.size} grid points...")
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = calculate_outlier_pi_by_investment(X[i, j], Y[i, j], age_cohort, age_outlier)

# Generate 3D surface plot to visualize sensitivity of pi to cohort size and wealth
plt.style.use('dark_background')

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X, Y, Z, cmap='plasma', edgecolor='navy')

ax.set_xlabel('Cohort Size (members)', labelpad=10)
ax.set_ylabel("Outlier's Investment ($)", labelpad=10)
ax.set_zlabel("Outlier's Participation Rate (pi)", labelpad=10)
ax.set_title(f'Participation Rates for Age {age_outlier} Single Outlier vs Age {age_cohort} Homogeneous Cohort', fontsize=14)

fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10)
plt.tight_layout()
plt.show()